# Collecting Daily Pitching

## Init

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import numpy as np
from io import StringIO
from datetime import date, timedelta

## Daily Tracker

In [0]:
lookup_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print("Fetching data from:", lookup_date)

## Retrieving Pitches From Baseball Savant

### Using Attackzone Selection csv

In [0]:
print("Starting pitch fetch")

# List of all attack zones
attack_zone = [nums for nums in range(1,40) if nums not in [10,15,20,25,30,35]]

pitch_df = pd.DataFrame()

url_csv = "https://baseballsavant.mlb.com/statcast_search/csv"

for zone in attack_zone:
    params_az = {
        "hfGT": "R|",
        "hfNewZones": zone,
        "hfSea": "2026|",
        "player_type": "pitcher",
        "game_date_gt": lookup_date,
        "game_date_lt": lookup_date,
        "group_by": "name",
        "min_pitches": 0,
        "min_results": 0,
        "min_pas": 0,
        "sort_col": "pitches",
        "sort_order": "desc",
        "type": "details",
        "all": "true",
        "minors": "false",
        "wbc": "false"
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "X-Requested-With": "XMLHttpRequest"
    }

    response = requests.get(url_csv, params=params_az, headers=headers, timeout=25)
    time.sleep(random.uniform(9.84, 15.67))

    df = pd.read_csv(StringIO(response.text))

    # Creating attack zone column
    df["attack_zone"] = zone
    
    pitch_df = pd.concat([pitch_df, df], ignore_index=True)

print(f"Total pitches for {lookup_date}: {len(pitch_df)}")
print("Pitch fetch complete")

## Small Cleaning and Appending to Bronze Schema

In [0]:
# Checking and removing auto balls, auto strikes, and intentional walks
pitch_df = pitch_df[~pitch_df['pitch_type'].isin(['AB', 'AS', 'IBB'])].copy()

print("Total pitches for", lookup_date, ": ", len(pitch_df))

spark_pitch_df = spark.createDataFrame(pitch_df)

print("Appending to Bronze Layer")

spark_pitch_df.write.mode("append").saveAsTable("pitch_data_2026.bronze.pitch_data")